In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier   # or KerasRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [10]:
%pip install scikeras
from sklearn.model_selection import train_test_split


Note: you may need to restart the kernel to use updated packages.


In [11]:
df=pd.read_csv('Churn_Modelling.csv')
df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
df['Gender']=label_encoder_gender.fit_transform(df['Gender'])

onehot_encoder_geography=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geography.fit_transform(df[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geography.get_feature_names_out(['Geography']))

df=pd.concat([df.drop('Geography',axis=1),geo_encoded_df],axis=1)
X=df.drop('Exited',axis=1)
y=df['Exited']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [7]:
import pickle

In [12]:
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)
with open('onehot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geography, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [20]:
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation='relu'))
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))   # use Dense(1) for regression
    return model

In [21]:
model = KerasClassifier(
    model=create_model,
    loss="binary_crossentropy",   # "mse" for regression
    optimizer="adam",
    metrics=["accuracy"],         # drop or use ["mae"] for regression
    model__neurons=32,
    model__layers=1,
    epochs=50,
    batch_size=10,
    verbose=0
)

In [22]:
param_grid = {
    'model__layers': [1, 2],
    'model__neurons': [16, 32],
    'epochs': [50, 100],       # no prefix: these belong to the wrapper itself
    'batch_size': [10, 20],
}

In [25]:
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.layers import Dense, Input

In [ ]:
grid=GridSearchCV(estimator=model,param_grid=param_grid,cv=3,n_jobs=-1)
grid_result=grid.fit(X_train,y_train)